### Task 8: Building a Collaborative Filtering Recommendation Model

#### Objective

The objective of this task is to develop a User-Based Collaborative Filtering Recommendation Model that recommends TED Talks based on user preferences and interaction patterns.

The model identifies users with similar rating patterns using Cosine Similarity and recommends TED Talks that similar users have rated highly.

#### Dataset

The user interaction dataset created in Task 7 will be used for this task.

Note: The user interaction data is simulated for educational and demonstration purposes.


### 1. Understanding User-Based Collaborative Filtering

#### What is Collaborative Filtering?

Collaborative Filtering is a recommendation technique that suggests items to a user based on the preferences and behaviour of other users with similar interests.

It assumes that users who liked similar items in the past may also like similar items in the future.

#### User-Based Collaborative Filtering

User-Based Collaborative Filtering identifies users with similar rating patterns and recommends items that similar users have rated highly.

#### Cosine Similarity

Cosine Similarity measures how similar two users are by comparing their rating patterns.

- A score closer to 1 indicates greater similarity.
- A score closer to 0 indicates lower similarity.

#### User-Item Matrix

A User-Item Matrix represents the ratings given by users to different TED Talks.

- Rows represent users.
- Columns represent TED Talks.
- Values represent user ratings.
- Missing values indicate that a user has not rated a talk.

#### Recommendation Process

1. Load the simulated user interaction dataset from Task 7.
2. Create a User-Item Matrix.
3. Calculate similarity between users using Cosine Similarity.
4. Identify users with similar preferences.
5. Find TED Talks that the target user has not rated.
6. Predict ratings using the ratings of similar users.
7. Recommend talks with the highest predicted ratings.

#### Difference Between Content-Based and Collaborative Filtering

| Content-Based Filtering | Collaborative Filtering |
|---|---|
| Uses item features such as title, description and tags. | Uses user ratings and interaction patterns. |
| Recommends similar content. | Recommends items liked by similar users. |
| Does not require other users' ratings. | Requires user interaction data. |

#### Limitations

- **Cold Start Problem:** New users or new items have insufficient interaction data.
- **Sparse Data:** Many users may not rate enough items.
- **Limited Interactions:** Recommendations may be restricted when users have rated most available items.
- **Simulated Data:** Results from simulated interactions may not represent real user preferences.


In [1]:

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load the simulated user interaction dataset from Task 7
interactions_df = pd.read_csv("../data/interaction/user_interactions.csv")

# Display dataset information
print("Dataset Shape:", interactions_df.shape)

display(interactions_df.head())

print("\nDataset Information:")
interactions_df.info()


Dataset Shape: (37, 7)


,user_id,talk_id,title,rating,liked,watched,bookmarked
0,U001,1,Averting the climate crisis,5,True,True,True
1,U001,92,The best stats you've ever seen,4,True,True,False
2,U001,7,Simplicity sells,3,False,True,False
3,U001,53,Greening the ghetto,5,True,True,True
4,U001,66,Do schools kill creativity?,2,False,True,False



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     37 non-null     str  
 1   talk_id     37 non-null     int64
 2   title       37 non-null     str  
 3   rating      37 non-null     int64
 4   liked       37 non-null     bool 
 5   watched     37 non-null     bool 
 6   bookmarked  37 non-null     bool 
dtypes: bool(3), int64(2), str(2)
memory usage: 1.4 KB


In [2]:

# Create a User-Item Matrix using ratings
user_item_matrix = interactions_df.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating"
)

# Display the matrix
print("User-Item Matrix:")
display(user_item_matrix)


User-Item Matrix:


talk_id,1,7,49,53,66,86,92,94
user_id,,,,,,,,
U001,5.0,3.0,4.0,5.0,2.0,5.0,4.0,3.0
U002,4.0,2.0,3.0,4.0,5.0,4.0,5.0,NaN
U003,2.0,5.0,5.0,4.0,3.0,4.0,3.0,2.0
U004,5.0,4.0,4.0,2.0,5.0,NaN,4.0,NaN
U005,3.0,4.0,2.0,5.0,3.0,4.0,5.0,5.0


In [3]:
# Replace missing ratings with 0 for similarity calculation

user_item_filled = user_item_matrix.fillna(0)

print("User-Item Matrix After Handling Missing Values:")
display(user_item_filled)


User-Item Matrix After Handling Missing Values:


talk_id,1,7,49,53,66,86,92,94
user_id,,,,,,,,
U001,5.0,3.0,4.0,5.0,2.0,5.0,4.0,3.0
U002,4.0,2.0,3.0,4.0,5.0,4.0,5.0,0.0
U003,2.0,5.0,5.0,4.0,3.0,4.0,3.0,2.0
U004,5.0,4.0,4.0,2.0,5.0,0.0,4.0,0.0
U005,3.0,4.0,2.0,5.0,3.0,4.0,5.0,5.0


In [4]:

# Calculate cosine similarity between users
user_similarity = cosine_similarity(user_item_filled)

# Create a DataFrame for the similarity matrix
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("User Similarity Matrix:")
display(user_similarity_df.round(3))


User Similarity Matrix:


user_id,U001,U002,U003,U004,U005
user_id,,,,,
U001,1.000,0.903,0.923,0.776,0.938
U002,0.903,1.000,0.868,0.874,0.852
U003,0.923,0.868,1.000,0.810,0.898
U004,0.776,0.874,0.810,1.000,0.732
U005,0.938,0.852,0.898,0.732,1.000


In [5]:
# Select a target user
target_user = "U001"

# Find users similar to the target user
similar_users = (
    user_similarity_df[target_user]
    .drop(target_user)
    .sort_values(ascending=False)
)

print(f"Users most similar to {target_user}:")
display(similar_users)


Users most similar to U001:


user_id
U005    0.937984
U003    0.923464
U002    0.902542
U004    0.775881
Name: U001, dtype: float64

In [6]:
#Build the Recommendation Function

def recommend_for_user(user_id, top_n=5):

    # Check whether the user exists
    if user_id not in user_item_matrix.index:
        print(f"User ID '{user_id}' was not found.")
        return pd.DataFrame(columns=["talk_id", "predicted_rating"])

    # Validate top_n
    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n <= 0:
        print("top_n must be a positive integer.")
        return pd.DataFrame(columns=["talk_id", "predicted_rating"])

    # Get the target user's ratings
    user_ratings = user_item_matrix.loc[user_id]

    # Find talks not yet rated by the target user
    unrated_talks = user_ratings[user_ratings.isna()].index

    # Find other users, sorted by similarity
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
    )

    recommendations = []

    # Calculate predicted ratings for unrated talks
    for talk_id in unrated_talks:

        # Get ratings given by other users to this talk
        ratings_for_talk = user_item_matrix[talk_id].dropna()

        # Exclude the target user
        ratings_for_talk = ratings_for_talk.drop(
            index=user_id, errors="ignore"
        )

        # Find users who both have a similarity score and rated this talk
        common_users = similar_users.index.intersection(
            ratings_for_talk.index
        )

        if len(common_users) == 0:
            continue

        # Calculate weighted average rating
        weights = similar_users.loc[common_users]
        ratings = ratings_for_talk.loc[common_users]

        weight_sum = weights.sum()

        if weight_sum <= 0:
            continue

        predicted_rating = (weights * ratings).sum() / weight_sum

        recommendations.append({
            "talk_id": talk_id,
            "predicted_rating": predicted_rating
        })

    # Convert recommendations into a DataFrame
    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return pd.DataFrame(columns=["talk_id", "predicted_rating"])

    # Sort by predicted rating and return top N
    return (
        recommendations_df
        .sort_values(by="predicted_rating", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


In [7]:
# Test recommendations for three different users

test_users = ["U002", "U004", "U001"]

for user in test_users:
    print(f"\nRecommendations for {user}:")
    recommendations = recommend_for_user(user, top_n=5)

    if recommendations.empty:
        print("No new recommendations available.")
    else:
        display(recommendations)



Recommendations for U002:


,talk_id,predicted_rating
0,94,3.319202



Recommendations for U004:


,talk_id,predicted_rating
0,86,4.243067
1,94,3.282451



Recommendations for U001:
No new recommendations available.


In [8]:
#Display TED Talk Titles Along With Recommendations

# Load the cleaned TED Talks dataset
ted_df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Create a lookup table for TED Talk IDs and titles
talk_lookup = ted_df[["talk_id", "title"]].drop_duplicates(
    subset="talk_id"
)

print("TED Talks dataset loaded successfully.")
display(talk_lookup.head())


TED Talks dataset loaded successfully.


,talk_id,title
0,1,Averting the climate crisis
1,92,The best stats you've ever seen
2,7,Simplicity sells
3,53,Greening the ghetto
4,66,Do schools kill creativity?


In [9]:
#Display Final Recommendations With Titles


# Generate recommendations for different users
for user in ["U002", "U004", "U001"]:

    print(f"\nFinal Recommendations for {user}:")

    recommendations = recommend_for_user(user, top_n=5)

    if recommendations.empty:
        print("No new recommendations available.")
    else:
        # Join recommendation results with TED Talk titles
        final_recommendations = recommendations.merge(
            talk_lookup,
            on="talk_id",
            how="left"
        )

        # Arrange columns for clear presentation
        final_recommendations = final_recommendations[
            ["talk_id", "title", "predicted_rating"]
        ]

        display(final_recommendations.round(3))



Final Recommendations for U002:


,talk_id,title,predicted_rating
0,94,Let's teach religion -- all religion -- in sch...,3.319



Final Recommendations for U004:


,talk_id,title,predicted_rating
0,86,Letting go of God,4.243
1,94,Let's teach religion -- all religion -- in sch...,3.282



Final Recommendations for U001:
No new recommendations available.


In [10]:
#Test Invalid User IDs and Invalid Inputs

# Test an invalid user ID
print("Testing invalid user ID:")
display(recommend_for_user("U999"))

# Test invalid top_n values
print("\nTesting invalid top_n values:")

display(recommend_for_user("U002", top_n=0))
display(recommend_for_user("U002", top_n=-2))
display(recommend_for_user("U002", top_n="five"))


Testing invalid user ID:
User ID 'U999' was not found.


,talk_id,predicted_rating



Testing invalid top_n values:
top_n must be a positive integer.


,talk_id,predicted_rating


top_n must be a positive integer.


,talk_id,predicted_rating


top_n must be a positive integer.


,talk_id,predicted_rating


In [11]:
#Analyze the Recommendation Results


# Analyze recommendation results for all test users

test_users = ["U002", "U004", "U001"]

for user in test_users:

    recommendations = recommend_for_user(user, top_n=5)

    print(f"\nUser: {user}")
    print("Number of recommendations:", len(recommendations))

    if not recommendations.empty:
        print(
            "Highest predicted rating:",
            round(recommendations["predicted_rating"].max(), 3)
        )
        print("Recommended Talk IDs:")
        print(recommendations["talk_id"].tolist())
    else:
        print("No unrated talks with available ratings from other users.")



User: U002
Number of recommendations: 1
Highest predicted rating: 3.319
Recommended Talk IDs:
[94]

User: U004
Number of recommendations: 2
Highest predicted rating: 4.243
Recommended Talk IDs:
[86, 94]

User: U001
Number of recommendations: 0
No unrated talks with available ratings from other users.



### 2. Observations and Results

1. A User-Item Matrix was created using the simulated user interaction dataset from Task 7.
2. Missing ratings were temporarily replaced with zero for calculating cosine similarity.
3. Cosine Similarity was used to measure the similarity between users based on their rating patterns.
4. The User Similarity Matrix showed the similarity scores between all pairs of users.
5. The recommendation function identifies similar users and uses their ratings to predict ratings for talks not yet rated by the target user.
6. Recommendations are sorted in descending order of predicted rating.
7. Different users can receive different recommendations depending on their existing ratings and the preferences of similar users.
8. Some users may receive no recommendations if they have already rated all available talks with recorded interactions.
9. The recommendation results demonstrate how collaborative filtering can use the preferences of multiple users to suggest relevant content.
10. Since the interaction dataset is simulated and small, the results are for demonstration purposes and may not represent real-world user preferences.

### 3. Conclusion

The User-Based Collaborative Filtering model was successfully developed using Cosine Similarity.

The model identifies similar users, predicts ratings for unrated TED Talks using similarity-weighted ratings, and recommends talks with the highest predicted scores.

This task demonstrates how user interaction data can be used to build a personalized recommendation system.



## 4. Answers to Conceptual Questions

### Q1. What is Collaborative Filtering?

Collaborative Filtering is a recommendation technique that suggests items to users based on the preferences and behaviour of other users with similar interests.

### Q2. What is User-Based Collaborative Filtering?

User-Based Collaborative Filtering identifies users with similar rating patterns and recommends items that those similar users have liked or rated highly.

### Q3. What is Cosine Similarity?

Cosine Similarity is a mathematical measure used to calculate the similarity between two vectors. In recommendation systems, it compares user rating patterns. A value closer to 1 indicates greater similarity.

### Q4. Why is a User-Item Matrix important?

A User-Item Matrix organizes user ratings in a structured format. Rows represent users, columns represent items, and values represent ratings. It helps calculate similarities between users.

### Q5. How are similar users identified?

Similar users are identified by calculating Cosine Similarity between their rating vectors. Users with higher similarity scores have more similar preferences.

### Q6. How are recommendations generated?

The system identifies similar users, finds talks that the target user has not rated, calculates predicted ratings using similarity-weighted ratings from other users, and recommends talks with the highest predicted scores.

### Q7. What is the Cold Start Problem?

The Cold Start Problem occurs when a recommendation system has insufficient information about a new user or a new item. Without enough interaction data, generating accurate recommendations becomes difficult.

### Q8. What are the limitations of Collaborative Filtering?

Some limitations include sparse interaction data, the cold start problem, scalability challenges, and dependence on user ratings and interaction history.

### Q9. How is Collaborative Filtering different from Content-Based Filtering?

Content-Based Filtering recommends items using their features, such as titles, descriptions, and tags. Collaborative Filtering recommends items using the preferences and rating patterns of similar users.

### Q10. How can Collaborative Filtering improve personalized recommendations?

It can improve personalization by learning from the preferences of users with similar interests. This helps discover items that a target user may enjoy even if those items have not been recommended to them before.


#### Creating the Task 8 PDF Observation Report

In [12]:

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
import os

# Create reports folder if it does not exist
os.makedirs("../reports", exist_ok=True)

# Set PDF file path
pdf_path = "../reports/task_08_observations.pdf"

# Create PDF document
doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50,
    leftMargin=50,
    topMargin=50,
    bottomMargin=50
)

styles = getSampleStyleSheet()
story = []

# Title
story.append(Paragraph(
    "Task 8: User-Based Collaborative Filtering",
    styles["Title"]
))
story.append(Spacer(1, 15))

# Report content
report_sections = [
    ("1. Objective",
     "To build a User-Based Collaborative Filtering recommendation model "
     "using Cosine Similarity and simulated user interaction data."),

    ("2. Dataset",
     "The simulated user interaction dataset created in Task 7 was used. "
     "It contains user IDs, TED Talk IDs, ratings, and interaction details."),

    ("3. Methodology",
     "A User-Item Matrix was created from user ratings. Missing ratings "
     "were temporarily filled with zero for similarity calculation. "
     "Cosine Similarity was then used to calculate similarities between users."),

    ("4. Recommendation Model",
     "A reusable recommend_for_user() function was developed. It identifies "
     "similar users, excludes talks already rated by the target user, and "
     "calculates predicted ratings using similarity-weighted averages."),

    ("5. Observations",
     "The model generates recommendations based on the rating patterns of "
     "similar users. Users with different unrated talks may receive different "
     "recommendations. Users who have rated all available talks with recorded "
     "interactions may receive no recommendations."),

    ("6. Limitations",
     "The dataset is small and simulated. Missing ratings are represented "
     "as zero only during similarity calculation, which can influence similarity "
     "scores. The model also faces sparse-data and cold-start limitations."),

    ("7. Conclusion",
     "The User-Based Collaborative Filtering model was successfully developed "
     "using Cosine Similarity. It demonstrates how user interaction data can "
     "support personalized TED Talk recommendations.")
]

for heading, content in report_sections:
    story.append(Paragraph(heading, styles["Heading2"]))
    story.append(Spacer(1, 6))
    story.append(Paragraph(content, styles["BodyText"]))
    story.append(Spacer(1, 12))

# Generate PDF
doc.build(story)

print("PDF report created successfully!")
print("Saved at:", pdf_path)


PDF report created successfully!
Saved at: ../reports/task_08_observations.pdf
